# Recess Ready: Philadelphia

**Question:** each evening, can we predict whether tomorrow will be unsafe for outdoor recess at Philadelphia schools, and **why** (heat, cold, bad air, ozone smog, flash-flood rain, dangerous wind, thunderstorms, heavy snow, or ice)?

*Prototype, not official guidance. Always follow official National Weather Service alerts and your school district.*

## Step 1: Download Philadelphia's daily weather and air quality

We ask Open-Meteo (free, no API key) for one row per day in Philadelphia:
- **Weather** (archive API): hottest and coldest "feels like" temperature, rain, snow, strongest wind gust, and a **weather code**. The weather code is a number that says the worst weather of the day, for example 95 = thunderstorm and 66 = freezing rain.
- **Air quality** (air-quality API): PM2.5 (tiny particles) and ozone, every hour. We turn the hours into one number per day.

Note: Open-Meteo's air quality numbers are **estimates from a computer model (CAMS)**, not readings from a sensor. We say this in our write-up.

In [ ]:
import pandas as pd
import requests

LAT, LON = 39.95, -75.16            # Philadelphia (Center City)
START, END = "2022-08-01", "2026-11-12"

# Weather: one row per day
weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather = requests.get(weather_url, params={
    "latitude": LAT, "longitude": LON,
    "start_date": START, "end_date": END,
    "daily": "apparent_temperature_max,apparent_temperature_min,precipitation_sum,"
             "snowfall_sum,wind_gusts_10m_max,weather_code",
    "timezone": "America/New_York",
}).json()
weather_df = pd.DataFrame(weather["daily"])
weather_df["time"] = pd.to_datetime(weather_df["time"])

# Air quality: one row per hour
air_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
air = requests.get(air_url, params={
    "latitude": LAT, "longitude": LON,
    "start_date": START, "end_date": END,
    "hourly": "pm2_5,ozone",
    "timezone": "America/New_York",
}).json()
air_df = pd.DataFrame(air["hourly"]).set_index("time")
air_df.index = pd.to_datetime(air_df.index)

# PM2.5: average of the day's 24 hours
pm_daily = air_df["pm2_5"].resample("D").mean()
# Ozone: the EPA uses 8-hour averages, so we take the worst 8-hour average of each day
ozone_daily = air_df["ozone"].rolling(8).mean().resample("D").max()
air_daily = pd.DataFrame({"pm2_5": pm_daily, "ozone_8h": ozone_daily}).reset_index()

# Put weather and air side by side, one row per day
df = weather_df.merge(air_daily, on="time")
print(df.shape)
df.head()

## Step 2: How many unsafe *school* days are there, and why?

A day is **unsafe** if any one of these happens. ⚠️ **Team to-do:** look up and cite a real source for each threshold (e.g. NWS, US EPA, or our school district's recess policy), and change the number if the source says something different.

| Warning | Rule |
|---|---|
| Too hot | "feels like" max ≥ 32 °C (≈90 °F, NWS "extreme caution") |
| Too cold | "feels like" min ≤ −7 °C (≈20 °F) |
| Bad air (smoke/particles) | daily PM2.5 > 35.4 µg/m³ (US EPA "Unhealthy for Sensitive Groups") |
| Ozone smog | worst 8-hour ozone > 137 µg/m³ (≈70 ppb, US EPA standard) |
| Flash flood / heavy rain | ≥ 25 mm (≈1 inch) of rain in the day |
| Dangerous wind (hurricanes, storms) | gusts ≥ 74 km/h (≈46 mph, NWS wind advisory level) |
| Thunderstorm / lightning | weather code 95, 96 or 99 |
| Heavy snow | ≥ 5 cm (≈2 inches) of snow |
| Ice / freezing rain | weather code 56, 57, 66 or 67 |

**Left out on purpose:** tornadoes (not in our data, very rare in Philadelphia; "dangerous wind" and "thunderstorm" cover the storms that bring them), earthquakes (nobody can predict them, and they aren't climate), and UV/sunburn (not in our weather data).

We only count **school days**: weekdays from September to June (no recess in July and August). We ignore holidays for now.

**Why this check comes first:** if a warning almost never happens, the model can't learn it. We'd drop it or change the threshold now, not on Sunday.

In [ ]:
df["too_hot"] = df["apparent_temperature_max"] >= 32
df["too_cold"] = df["apparent_temperature_min"] <= -7
df["bad_air"] = df["pm2_5"] > 35.4
df["ozone_smog"] = df["ozone_8h"] > 137
df["heavy_rain"] = df["precipitation_sum"] >= 25
df["high_wind"] = df["wind_gusts_10m_max"] >= 74
df["thunderstorm"] = df["weather_code"].isin([95, 96, 99])
df["heavy_snow"] = df["snowfall_sum"] >= 5
df["ice"] = df["weather_code"].isin([56, 57, 66, 67])

hazards = ["too_hot", "too_cold", "bad_air", "ozone_smog", "heavy_rain",
           "high_wind", "thunderstorm", "heavy_snow", "ice"]
df["unsafe"] = df[hazards].any(axis=1)   # True if ANY hazard happens

is_weekday = df["time"].dt.weekday < 5           # Monday=0 ... Friday=4
is_school_month = ~df["time"].dt.month.isin([7, 8])
school = df[is_weekday & is_school_month].copy()
school["year"] = school["time"].dt.year

print("School days:", len(school))
print("Unsafe school days:", school["unsafe"].sum())
print()
print("Unsafe school days per year, by reason:")
print(school.groupby("year")[hazards + ["unsafe"]].sum().T)   # .T flips the table so it fits

## Step 3: Turn each day into a question about *tomorrow*

Right now each row is one day. We want: "using **today's** weather and air, predict **tomorrow's** dangers."

- `features` = what we know today (the model's inputs).
- `shift(-1)` moves tomorrow's answers up one row, so each row holds today's inputs **and** tomorrow's answers side by side.
- We keep only rows where **tomorrow is a school day**.

**Leakage check:** the inputs are only from today. Nothing from tomorrow goes into `features`.

In [ ]:
df = df.sort_values("time").reset_index(drop=True)
df["month"] = df["time"].dt.month

features = ["apparent_temperature_max", "apparent_temperature_min", "precipitation_sum",
            "snowfall_sum", "wind_gusts_10m_max", "weather_code", "pm2_5", "ozone_8h", "month"]

# Tomorrow's answers, moved up one row
target_cols = []
for h in hazards:
    df[h + "_tomorrow"] = df[h].shift(-1)
    target_cols.append(h + "_tomorrow")

df["tomorrow"] = df["time"] + pd.Timedelta(days=1)
tomorrow_is_school = (df["tomorrow"].dt.weekday < 5) & ~df["tomorrow"].dt.month.isin([7, 8])

data = df[tomorrow_is_school].dropna(subset=features + target_cols).copy()
data[target_cols] = data[target_cols].astype(bool)
data["unsafe_tomorrow"] = data[target_cols].any(axis=1)
print("Rows we can learn from:", len(data))

## Step 4: Split by time (never shuffle)

- **Train** on every school day **before** September 2025.
- **Test** on the **2025-26 school year**, which the model never sees while learning.

This is like a real forecast: you only ever know the past.

In [ ]:
TEST_START = "2025-09-01"
train = data[data["tomorrow"] < TEST_START]
test = data[data["tomorrow"] >= TEST_START]

print("Train:", train["tomorrow"].min().date(), "to", train["tomorrow"].max().date(), "-", len(train), "days")
print("Test: ", test["tomorrow"].min().date(), "to", test["tomorrow"].max().date(), "-", len(test), "days")
print("Unsafe share in train:", round(train["unsafe_tomorrow"].mean(), 2))
print("Unsafe share in test: ", round(test["unsafe_tomorrow"].mean(), 2))

## Step 5: The baseline, "tomorrow will be the same as today"

The simplest possible guess: whatever danger happened today, guess it happens again tomorrow. Our model is only useful if it beats this.

In [ ]:
baseline_pred = test[hazards].values.astype(bool)   # today's dangers = guess for tomorrow

## Step 6: Train the model (a random forest)

A **random forest** is a group of decision trees. Each tree asks yes/no questions like "is today's 'feels like' max above 30 °C?" and the trees vote.

- We train **one forest that predicts all 9 dangers at once** (scikit-learn allows several answers per row).
- `class_weight="balanced"` tells the model that rare unsafe days matter, so it doesn't just say "safe" every time.
- `random_state=42` makes the result the same every time we run it.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                               class_weight="balanced", random_state=42)
model.fit(train[features], train[target_cols])
model_pred = model.predict(test[features]).astype(bool)

## Step 7: Model vs. baseline

For each danger we count, on the test year:
- **caught**: dangerous days we warned about (higher is better)
- **missed**: dangerous days we did NOT warn about (the worst mistake for kids)
- **false alarms**: we warned but it was fine (annoying, less harmful)

⚠️ If accuracy is above 95%, be suspicious and look for leakage.

In [ ]:
y_true = test[target_cols].values.astype(bool)

def score(true, pred):
    return {"caught": int((true & pred).sum()),
            "missed": int((true & ~pred).sum()),
            "false alarms": int((~true & pred).sum())}

rows = []
for i, h in enumerate(hazards):
    b = score(y_true[:, i], baseline_pred[:, i])
    m = score(y_true[:, i], model_pred[:, i])
    rows.append({"danger": h, "real days": int(y_true[:, i].sum()),
                 "baseline caught": b["caught"], "baseline missed": b["missed"], "baseline false alarms": b["false alarms"],
                 "model caught": m["caught"], "model missed": m["missed"], "model false alarms": m["false alarms"]})
print(pd.DataFrame(rows).to_string(index=False))

# Overall: is tomorrow unsafe for ANY reason?
true_any = y_true.any(axis=1)
base_any = baseline_pred.any(axis=1)
model_any = model_pred.any(axis=1)
print()
for name, pred in [("Baseline", base_any), ("Model", model_any)]:
    s = score(true_any, pred)
    acc = (pred == true_any).mean()
    print(f"{name}: accuracy {acc:.2f} | caught {s['caught']} | missed {s['missed']} | false alarms {s['false alarms']}")
    if acc > 0.95:
        print("   WARNING: above 95%. Check for leakage before trusting this!")

## Step 8: Confusion matrix (overall "unsafe tomorrow")

A 2x2 table: rows are what really happened, columns are what the model said.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(true_any, model_any, display_labels=["Safe", "Unsafe"], cmap="Blues")
plt.title("Recess Ready: model on the 2025-26 school year")
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 9: Where does it fail?

Real unsafe days the model **missed**. We look at a few and ask why.

In [ ]:
missed = test[true_any & ~model_any]
print("Missed unsafe days:", len(missed))
for _, row in missed.head(3).iterrows():
    real = [h for h in hazards if row[h + "_tomorrow"]]
    print(f"- {row['tomorrow'].date()}: really {real}. "
          f"Today's feels-like max {row['apparent_temperature_max']:.0f} C, min {row['apparent_temperature_min']:.0f} C, "
          f"PM2.5 {row['pm2_5']:.0f}, gusts {row['wind_gusts_10m_max']:.0f} km/h")

## Step 10: Does it work equally well in every season?

Our fairness check: the same accuracy and missed-day counts, split by season.

In [ ]:
def season(month):
    if month in [9, 10, 11]: return "Fall"
    if month in [12, 1, 2]: return "Winter"
    if month in [3, 4, 5]: return "Spring"
    return "June"

seasons = test["tomorrow"].dt.month.map(season).values
rows = []
for s in ["Fall", "Winter", "Spring", "June"]:
    keep = seasons == s
    t, m, b = true_any[keep], model_any[keep], base_any[keep]
    rows.append({"season": s, "days": int(keep.sum()), "unsafe days": int(t.sum()),
                 "model accuracy": round((m == t).mean(), 2),
                 "baseline accuracy": round((b == t).mean(), 2),
                 "model missed": int((t & ~m).sum())})
print(pd.DataFrame(rows).to_string(index=False))

## Step 11: The demo app

Pick a school day from the test year. The app shows what the model warned the evening before, and what really happened.

*Prototype, not official guidance. Always follow official National Weather Service alerts and your school district.*

In [ ]:
import gradio as gr

NAMES = {"too_hot": "🔥 Too hot", "too_cold": "🥶 Too cold", "bad_air": "😷 Bad air (smoke/particles)",
         "ozone_smog": "🌫️ Ozone smog", "heavy_rain": "🌊 Heavy rain / flash flood",
         "high_wind": "🌀 Dangerous wind", "thunderstorm": "⚡ Thunderstorm / lightning",
         "heavy_snow": "❄️ Heavy snow", "ice": "🧊 Ice / freezing rain"}

test_dates = test["tomorrow"].dt.strftime("%Y-%m-%d").tolist()

def recess_check(date_str):
    idx = test.index[test["tomorrow"].dt.strftime("%Y-%m-%d") == date_str][0]
    pred = model.predict(test.loc[[idx], features]).astype(bool)[0]
    warned = [NAMES[h] for h, p in zip(hazards, pred) if p]
    happened = [NAMES[h] for h in hazards if test.loc[idx, h + "_tomorrow"]]

    if warned:
        msg = "## 🔴 Plan indoor recess\n**Why:** " + ", ".join(warned)
    else:
        msg = "## 🟢 Outdoor recess looks OK"
    msg += "\n\n**What really happened:** " + (", ".join(happened) if happened else "a safe day")
    msg += "\n\n*Prototype, not official guidance. Always follow NWS alerts and your school district.*"
    return msg

demo = gr.Interface(fn=recess_check,
                    inputs=gr.Dropdown(test_dates, label="School day (2025-26 test year)"),
                    outputs=gr.Markdown(),
                    title="Recess Ready: Philadelphia",
                    description="The evening before: is tomorrow safe for outdoor recess?")
demo.launch()